# 10 · A case of your own

Chapter 10 takes four things you write outside the package, an accounting spec, a parameters file, a feed and a case file, and returns a six-stage run of your case, whose sidecars name the file it came from.

In [1]:
import hashlib
import inspect
import json
from pathlib import Path

import pandas as pd

from pysuto.registry import ConstraintRunRegistry, ImputationRunRegistry, PsutRunRegistry
from pysuto.tutorial import toy_ext

RUNS = Path("runs")
DATA = RUNS / "my_case"
TEMPLATE = Path("params_template.csv")
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.width", 200)

Chapters 00–09 ran the toy, a case that ships inside the package. A case is only a declaration, so you can write one in a file of your own. Nothing below is written into the package: every file goes into this folder, and the run goes to `runs/my_case/`.

The case is one mill in one region, Aland, over three years. Ore comes from a deposit into the Mill. The Mill makes a concentrate, which is shipped to a user, and tailings, which are not. Masses are in `kt` (gross) and `kt Content` (the metal in them). One gauge at the mill's gate reports the ore feed in both units. That is all the data there is.

## 1 · The accounting structure

The accounting structure lists the places a flow can start and end, and which place may send to which. You write it as a dict, an *acc spec* (ADR 0046), and register it as `input/acc/<id>/current/acc.json`. A `role` binds your label to the framework's role: a `nature` boundary, a `final_demand` boundary, a `gross` and a `content` valuation. The library reads roles, never your names (ADR 0034).

In [2]:
ACC = {
    "sectors": [{"name": "Deposit", "entity": "SB", "sub": "Raw", "role": "nature"},
                {"name": "Mill", "entity": "Process"},
                {"name": "Concentrate", "entity": "Flow"},
                {"name": "Tailings", "entity": "SB", "sub": "Sec", "role": "secondary"},
                {"name": "Use", "entity": "SB", "sub": "Demand", "role": "final_demand"}],
    "regions": ["Aland"],
    "valuations": [{"name": "Gross", "role": "gross"}, {"name": "Content", "role": "content"}],
    "times": ["2001", "2002", "2003"],
    "supply": {"Mill": ["Concentrate", "Tailings"]},
    "use": {"Deposit": ["Mill"], "Concentrate": ["Use"]},
    "tradable": [],
    "accounting": [{"from": "Process", "to": "Flow", "same_region": True},
                   {"from": "Process", "to": "SB", "same_region": True},
                   {"from": "Flow", "to": "SB", "same_region": False},
                   {"from": "SB", "to": "Process", "same_region": True}],
    "masks": [{"mask_name": "massbal_scope", "logic": "include", "sector_from": "*", "sector_to": "*"}],
}
ACC_DIR = DATA / "input" / "acc" / "mycase" / "current"
ACC_DIR.mkdir(parents=True, exist_ok=True)
(ACC_DIR / "acc.json").write_text(json.dumps(ACC, indent=2))
print(f"{len(ACC['sectors'])} sectors, {len(ACC['regions'])} region, {len(ACC['times'])} years, "
      f"{len(ACC['valuations'])} valuations: written to {ACC_DIR / 'acc.json'}")

5 sectors, 1 region, 3 years, 2 valuations: written to runs/my_case/input/acc/mycase/current/acc.json


`supply` says which process makes which commodity, `use` which node feeds which. `masks` names the nodes whose mass balance Stage 3 enforces; here, all of them.

## 2 · The parameters

Chapter 03 used parameters files, each a family of rows in one schema. The schema ships with the package as `params_template.csv`, and `python -m pysuto tutorial` copied it next to this notebook. Its `#` lines are a column reference, and below it sit worked examples. The examples are copper's, so ignore their sectors. The start of the reference, and the ids of the examples:

In [3]:
print("\n".join(TEMPLATE.read_text(encoding="utf-8").splitlines()[1:9]))
print("worked examples:", pd.read_csv(TEMPLATE, sep=";", comment="#")["param_id"].tolist())

# ── COLUMN REFERENCE ──────────────────────────────────────────────────────────────────────────────────────
# param_id          Unique string key. MUST be underscore-free: it becomes the
#                   constraint-id `variant` slot, and `_` is the id separator
#                   (see constraints/naming._safe). Use e.g. effsmelting, conccosupply.
# param_type        concentration | efficiency | stripping_ratio
#                   stripping_ratio: a Mass→Mass ratio between two process OUTPUTS
#                   (WR_Mass = stripping_ratio·Ore_Mass at Mining); not a CC recovery
#                   fraction. Read by NetworkImputer (WR fill) + the ratio builders.
worked examples: ['conc_concentrate_use', 'conc_concentrate_supply', 'eff_sxew', 'eff_mf']


One row says: *edge A and edge B are tied by a coefficient*, `f_B = coefficient × f_A`. The `a_*` columns select the edges A, the `b_*` columns the edges B; `*` matches anything, and `same` in a `b_*` column copies what A matched. Our family, `eff`, has three kinds of row, each written for both valuations. The Mill's concentrate takes 90 % of what it is fed, its tailings take 10 %, and the concentrate is shipped in full. A parameter id carries no underscore (ADR 0005), because `_` separates the parts of a constraint id.

In [4]:
COLS = [c for c in pd.read_csv(TEMPLATE, sep=";", comment="#", nrows=0).columns if c[:2] in ("a_", "b_")]


def pair(name, a, b, value, sd, v):
    row = dict(zip(COLS, a + (v,) + b + ("same",)))
    return {"param_id": name + v.lower(), "param_type": "efficiency", **row,
            "value_nominal": value, "unc_type": "sd", "value_sd": sd, "source": "chapter 10"}

In [5]:
MILL_IN = ("Deposit", "SB", "*", "Mill", "Process", "same", "*")
MILL_OUT = ("Mill", "Process", "*", "Concentrate", "Flow", "same", "*")
TO_CONCENTRATE = ("Mill", "Process", "same", "Concentrate", "Flow", "same", "same")
TO_TAILINGS = ("Mill", "Process", "same", "Tailings", "SB", "same", "same")
TO_USE = ("Concentrate", "Flow", "same", "Use", "SB", "same", "same")
KINDS = (("effconcentrate", MILL_IN, TO_CONCENTRATE, 0.90, 0.02),
         ("efftailings", MILL_IN, TO_TAILINGS, 0.10, 0.02),
         ("effship", MILL_OUT, TO_USE, 1.00, 0.01))
ROWS = [pair(*kind, v) for v in ("Gross", "Content") for kind in KINDS]
PARAMS = DATA / "input" / "params" / "eff" / "current" / "params.csv"
toy_ext.params_from_template(TEMPLATE, ROWS, PARAMS)
pd.read_csv(PARAMS, sep=";")[["param_id", "a_sector_from", "a_sector_to", "b_sector_to", "a_valuation", "value_nominal", "value_sd"]]

,param_id,a_sector_from,a_sector_to,b_sector_to,a_valuation,value_nominal,value_sd
0,effconcentrategross,Deposit,Mill,Concentrate,Gross,0.900,0.020
1,efftailingsgross,Deposit,Mill,Tailings,Gross,0.100,0.020
2,effshipgross,Mill,Concentrate,Use,Gross,1.000,0.010
3,effconcentratecontent,Deposit,Mill,Concentrate,Content,0.900,0.020
4,efftailingscontent,Deposit,Mill,Tailings,Content,0.100,0.020
5,effshipcontent,Mill,Concentrate,Use,Content,1.000,0.010


Six rows, in the template's columns and none of its examples. The family's id is the folder it sits in, `eff`: the case file will ask for it by that name.

## 3 · The feed

A feed is three files in `input/feed/<id>/current/` (chapter 01): `values.csv`, the raw table in the feed's own vocabulary; `conc.xlsx`, the concordance that maps its labels to yours, one sheet per map; and `meta.xlsx`, its pedigree scores. The gauge reports the Mill's ore feed, one gross and one content reading a year. The numbers are made up.

In [6]:
FEED_DIR = DATA / "input" / "feed" / "gauge" / "current"
FEED_DIR.mkdir(parents=True, exist_ok=True)
RAW = pd.DataFrame({
    "Year": ["2001", "2002", "2003"] * 2, "Region": "Aland", "Material": "ore feed",
    "Unit": ["kt"] * 3 + ["kt Content"] * 3, "Value": [98.0, 103.5, 101.2, 1.47, 1.55, 1.52]})
RAW.to_csv(FEED_DIR / "values.csv", index=False)
RAW

,Year,Region,Material,Unit,Value
0,2001,Aland,ore feed,kt,98.000
1,2002,Aland,ore feed,kt,103.500
2,2003,Aland,ore feed,kt,101.200
3,2001,Aland,ore feed,kt Content,1.470
4,2002,Aland,ore feed,kt Content,1.550
5,2003,Aland,ore feed,kt Content,1.520


The concordance has one sheet per map. `sector_map`, `region_map`, `time_map` and `valuation_map` send the gauge's labels to ours (`Unit` `kt Content` is our `Content`), and `var_map` says which raw column plays which role. The pedigree scores (1 is best, 4 worst) set the feed's CV (Laner et al. 2016; chapter 01).

In [7]:
def sheet(key, pairs):
    return pd.DataFrame(pairs, columns=[f"src_{key}", f"root_{key}"])


TABLES = {"sector_map": sheet("sector", [("ore feed", "Deposit")]),
          "region_map": sheet("region", [("Aland", "Aland")]),
          "time_map": sheet("time", [(y, y) for y in ("2001", "2002", "2003")]),
          "valuation_map": sheet("valuation", [("kt", "Gross"), ("kt Content", "Content")]),
          "var_map": pd.DataFrame([("Year", "src_time"), ("Material", "src_sector_from"),
                                   ("Region", "src_region_from"), ("Region", "src_region_to"),
                                   ("Unit", "src_valuation")], columns=["raw_var", "src_var"])}
with pd.ExcelWriter(FEED_DIR / "conc.xlsx") as xl:
    for name, table in TABLES.items():
        table.to_excel(xl, sheet_name=name, index=False)

In [8]:
SCORES = {"Reliability": 2, "Completeness": 3, "Temporal_correlation": 2,
          "Geographical_correlation": 2, "Other_correlation": 2}
META = pd.DataFrame({"Grouping": "Unc", "Meta_name": list(SCORES), "Value": list(SCORES.values())})
META.loc[len(META)] = ["Ident.", "Feed_name", "gauge"]
META.to_excel(FEED_DIR / "meta.xlsx", index=False)
META

,Grouping,Meta_name,Value
0,Unc,Reliability,2
1,Unc,Completeness,3
2,Unc,Temporal_correlation,2
3,Unc,Geographical_correlation,2
4,Unc,Other_correlation,2
5,Ident.,Feed_name,gauge


## 4 · The case file

A case is a Python module with one `CaseSpec` called `CASE`. `%%writefile` saves a cell as a file, so the file is the cell. The first part declares the two sections of the case, one per stage that has knobs (chapters 02 and 03). A *knob* is an UPPER_CASE field: the stage reads it, the run records it, and `--set KNOB=VALUE` changes it. The values are the toy's.

In [9]:
%%writefile my_case.py
"""my_case.py: a minimal case, one mill in one region (chapter 10)."""

from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

import pandas as pd

from pysuto.case import CaseSpec, ConstSection, FillEntry, ImpSection
from pysuto.constraints import MassBalanceBuilder, RatioConstraintBuilder
from pysuto.feedpipe import FeedSpec
from pysuto.impute import FlatPriorImputer, TimeImputer
from pysuto.tutorial import toy_ext

ACC_ID = "mycase"


@dataclass(frozen=True)
class MyImp(ImpSection):
    P_PROP:        float = 1.3
    Q_REG:         float = 3.0
    ALPHA_MIN_REG: float = 0.05


@dataclass(frozen=True)
class MyConst(ConstSection):
    P_PROP:                float = 1.3
    SIGMA_A_CV_MIN:        float = 0.001
    SIGMA_A_ABS_FLOOR:     float = 1.0e-6
    SIGMA_C_CV_MIN:        float = 0.01
    SIGMA_C_ABS_FLOOR:     float = 1.0e-3
    PRIOR_BOX_K:           Optional[float] = None
    NODATA_ANCHOR_CV:      float = 0.05
    NODATA_ANCHOR_EXCLUDE: tuple = ()
    PIN_FLAT_PRIOR_ZEROS:  bool = False
    PIN_EXCLUDE_SECTORS:   tuple = ()
    DIAG_PLOT:             bool = False

Writing my_case.py


Next, the two functions that choose what runs. `terminal` returns the imputers that run once, after the cascade (chapter 02). `builders` returns the constraint builders of Stage 3 (chapter 03): the hard mass balance at every node of the `massbal_scope` mask, and one soft row per line of the `eff` file. `ctx.use_params("eff")` resolves the registered file and records its version as a parent of the run.

In [10]:
%%writefile -a my_case.py


def terminal(ctx) -> list:
    flat = FlatPriorImputer(acc_struct=ctx.acc, mode="noisy_zero", sigma_high=1.0e6)
    return [TimeImputer(id="time_interp"), flat]


def builders(ctx) -> list:
    eff = RatioConstraintBuilder.from_csv(ctx.use_params("eff"), acc_struct=ctx.acc,
                                          sigma_rel_floor=0.02, scale_abs_floor=1.0)
    return [MassBalanceBuilder(mask="massbal_scope"), eff]

Appending to my_case.py


Now the feed's side of the file. A `FeedSpec` is Stage 1's public entry (chapter 01): it names the feed and says how to load its two tables, and `pysuto.feedpipe.run_feed` does the rest. It reads the three registered files, builds the concordance and the weight matrix, propagates the pedigree to a σ for every edge and registers the feed run.

In [11]:
%%writefile -a my_case.py


def load_raw(path):
    return pd.read_csv(path, dtype={"Year": str})


def load_tables(path):
    return pd.read_excel(path, sheet_name=None, dtype=str)


GAUGE = FeedSpec("gauge", load_raw=load_raw, load_tables=load_tables)

Appending to my_case.py


Last, the `CaseSpec`. It names the case and its accounting structure, hands over the two sections, and carries `stage1`, the hook that runs Stage 1: it builds the accounting structure from the registered spec and runs each feed spec through `run_feed`. The cascade has one entry, a fill that reads the `eff` family in both directions: from an observed edge it writes the edges the rows tie to it. Here is the hook's code.

In [12]:
print(inspect.getsource(toy_ext.register_feeds))

def register_feeds(data_dir, acc_id: str, specs: "list[FeedSpec]") -> "dict[str, str]":
    """Stage 1 of a case of your own, through the public API: build the accounting
    structure from the acc spec registered under ``<data_dir>/input/acc/<acc_id>`` (the
    structure and its build record go to ``<data_dir>/proc/acc``), then run each
    :class:`~pysuto.feedpipe.FeedSpec` through :func:`~pysuto.feedpipe.run_feed`, which
    reads the feed's registered ``values.csv``, ``conc.xlsx`` and ``meta.xlsx``.
    Returns ``{feed_id: feed_run_id}``."""
    data_dir  = Path(data_dir)
    inputs    = InputRegistries.initialize(root=str(data_dir / "input"))
    acc_dir   = data_dir / "proc" / "acc"
    spec      = read_acc_spec_dir(inputs.acc.resolve(acc_id))
    acc_store.save(AccountingStructure.from_spec(spec, acc_id, storage_dir=str(acc_dir)))
    write_build_record(str(acc_dir), inputs.acc.entry(acc_id))
    return {s.feed_id: run_feed(s, acc_id=acc_id, data_dir=str(data_dir),
             

In [13]:
%%writefile -a my_case.py


CASE = CaseSpec(
    id="mycase", acc_id=ACC_ID,
    imp=MyImp(cascade=(FillEntry("eff", "eff", "both"),), terminal=terminal),
    const=MyConst(builders=builders),
    stage1=lambda data_dir: toy_ext.register_feeds(data_dir, ACC_ID, [GAUGE]),
)

Appending to my_case.py


In [14]:
print(f"my_case.py: {len(Path('my_case.py').read_text().splitlines())} lines")

my_case.py: 68 lines


## 5 · Run it

One command runs all six stages:

```
python -m pysuto run --case ./my_case.py --data-dir runs/my_case
```

`--case` takes a bundled name (`toy`), a dotted module (`mylab.zinc`) or a path. Each stage starts in a fresh child process that loads your file by path, so the file has to stand on its own. `toy_ext.run_case` runs that command with this notebook's interpreter and returns the run ids it printed.

In [15]:
ids = toy_ext.run_case("./my_case.py", DATA)
print(json.dumps(ids, indent=1))
print(sorted(p.name for p in (DATA / "logs" / "pipeline").iterdir()))

{
 "feed": "feed_gauge_20261123_151946_4ca070b8",
 "imp": "imp_20261123_151947_eb6b37e0",
 "const": "const_20261123_151951_ea9f9f24",
 "opt": "opt_20261123_151952_e7668112",
 "uncprop": "uncprop_20261123_152000_27b6ab99",
 "psut": "psut_20261123_152003_4bda3d46"
}
['mycase_const.log', 'mycase_feed.log', 'mycase_imp.log', 'mycase_opt.log', 'mycase_psut.log', 'mycase_uncprop.log']


Six stages, six run ids, and one log per stage: each ran in its own process. The runs are in `runs/my_case/proc/`, one folder per stage, each with a `metadata.json`, its sidecar.

## 6 · What the run says about itself

A run records which case made it. The optimiser run's sidecar holds `case_source`, where the case came from, and `case_sha256`, the hash of that file's bytes.

In [16]:
def sidecar(stage, run_id, root=DATA):
    return json.loads((root / "proc" / stage / run_id / "metadata.json").read_text())


case = sidecar("opt", ids["opt"])["stage_config"]["case"]
print(f"case_source: {case['case_source']}")
print(f"case_sha256: {case['case_sha256']}")
print(f"sha256 of my_case.py: {hashlib.sha256(Path('my_case.py').read_bytes()).hexdigest()}")

case_source: path:/path/to/tutorial/my_case.py
case_sha256: 98a2cfaac389c1b4f48a89bd4e8d79bdabb96d4acd101934759c8ec0c55a8208
sha256 of my_case.py: 98a2cfaac389c1b4f48a89bd4e8d79bdabb96d4acd101934759c8ec0c55a8208


`case_source` is `path:` and the absolute path of your file. The saved outputs of this chapter write your tutorial folder as `/path/to/tutorial`; yours prints the real one. The two hashes are equal. Change one character of `my_case.py`, run again, and the hash changes: a run names the file that made it, and you can check.

A case that is not resolved from `--case` has no file to name. The toy chain of chapters 00–09 was run from Python, and its optimiser run records only the case's id:

In [17]:
print(sidecar("opt", toy_ext.base_ids(RUNS)["opt"], RUNS)["stage_config"]["case"])

{'id': 'toy', 'acc_id': 'toy'}


Now the result. The PSUT of chapter 06 is one row per edge, valuation and year. This is 2002.

In [18]:
psut = PsutRunRegistry(str(DATA / "proc" / "psut")).load(ids["psut"])
table = psut.table
table[table["period"] == "2002"][["valuation", "block", "from_sector", "to_sector", "val"]]

,valuation,block,from_sector,to_sector,val
8,Content,E,Deposit,Mill,1.550
9,Content,P,Mill,Tailings,0.155
10,Content,S,Mill,Concentrate,1.395
11,Content,Y,Concentrate,Use,1.395
12,Gross,E,Deposit,Mill,103.500
13,Gross,P,Mill,Tailings,10.350
14,Gross,S,Mill,Concentrate,93.150
15,Gross,Y,Concentrate,Use,93.150


In [19]:
log = ImputationRunRegistry(str(DATA / "proc" / "imp")).load(ids["imp"]).imputation_log.as_dataframe()
rows = ConstraintRunRegistry(str(DATA / "proc" / "const")).load(ids["const"]).row_meta
print(log.groupby(["imputer_id", "param_id"], dropna=False).size().to_string())
print(rows.groupby(["type", "variant", "is_hard"]).size().to_string())

imputer_id  param_id             
eff         effconcentratecontent    3
            effconcentrategross      3
            effshipcontent           3
            effshipgross             3
            efftailingscontent       3
            efftailingsgross         3
type     variant                is_hard
massbal  massbal                True       12
ratio    effconcentratecontent  False       3
         effconcentrategross    False       3
         effshipcontent         False       3
         effshipgross           False       3
         efftailingscontent     False       3
         efftailingsgross       False       3


The gauge read 1.55 `kt Content` for the Mill's feed in 2002. The run books about 90 % of it into the concentrate, which is shipped in full to `Use`, and about 10 % into tailings, in both valuations. Every edge has a value, although the gauge saw only the Mill's feed. Where the other values come from is in the logs of Stages 2 and 3: the imputation log says which imputer wrote each edge, and the constraint rows say what Stage 3 holds the edges to.

**Exercise 10.1** — How much `Content` does the Mill book into its tailings in 2002? Take the Mill's feed from the table above and the share from the `efftailingscontent` row of your parameters file. Cover the next cell and try it first.

In [20]:
tailings = 1.55 * 0.10
print(toy_ext.check_10_1(table, tailings))

✓ 10.1  Mill → Tailings 2002: 0.155 kt Content


<details><summary>Going deeper: what the file has to be, and what the hash covers</summary>

Every stage runs in its own process (the six logs above), and each one loads `my_case.py` again by its path. Keep module-level code cheap and deterministic: a case that drew random numbers would draw them anew in every process, and the toy fixes one seed for that reason (`pysuto.tutorial.toy`).

`case_sha256` is the hash of the file's bytes, so a comment changes it too. It names the code. The data the code read is named elsewhere: `parents.inputs` of the imputation and constraint runs hold the version of each registered input (your acc spec, your `eff` file), and the registries bump a version when a file's content changes (ADR 0023). A run is reproducible from the pair: the file with that hash, and the inputs with those versions.

A case written in Python and handed to the stage functions directly has no file, and records no `case_source` (`CaseSpec.case_source` is set only by `resolve_case`).
</details>

## Where to go from here: the seams of chapter 09

Your next change is one of the three seams of chapter 09, and each has a place in `my_case.py`:

| seam | where it goes in your file |
| --- | --- |
| imputer (Seam 1) | the list `terminal` returns, in front of the `TimeImputer` |
| constraint builder (Seam 2) | the list `builders` returns |
| feed (Seam 3) | the list handed to `toy_ext.register_feeds`: `[GAUGE, SECOND]` |

The fourth, a posterior propagator, is not part of a case file: Stage 5 takes the optimiser run. Add a row to your `eff` file, a second feed or a builder, run the command again, and compare the two optimiser runs. Which number moved, and by how much, is the question every chapter since 04 taught you to ask.

## Where this run came from

The imputation run's sidecar lists the inputs it used with their versions, and the feed runs it aggregated.

In [21]:
parents = sidecar("imp", ids["imp"])["parents"]
print(json.dumps(parents["inputs"], indent=1))
print(json.dumps(parents["upstream"], indent=1))

{
 "acc": {
  "id": "mycase",
  "version": 1,
  "date": "2026-11-23T15:19:46Z",
  "code": "b51cd9c0",
  "label": "mycase"
 },
 "params": [
  {
   "id": "eff",
   "version": 1,
   "date": "2026-11-23T15:19:46Z",
   "code": "6f01d733",
   "label": "eff"
  }
 ]
}
{
 "feed_run_ids": [
  "feed_gauge_20261123_151946_4ca070b8"
 ]
}


The registered acc spec and the `eff` file each have a version, and the feed run is the gauge's. That is the end of the tutorial: you have run the toy and a case of your own through the same six stages.